In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from jarvis.core.kpoints import Kpoints3D
from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms
from matplotlib.gridspec import GridSpec
import re

#####extract the fermi level in eV####
fermi_file_path = "./detailed.out"

# Read file content
with open(fermi_file_path, 'r') as file:
    lines = file.readlines()

# Extract numeric value in eV using regex
fermi_eV = None
for line in lines:
    if 'Fermi' in line and 'eV' in line:
        match = re.search(r'([-+]?\d*\.\d+|\d+)\s*eV', line)
        if match:
            fermi_eV = float(match.group(1))
        break


# File path
file_path = "band_tot.dat"
data = pd.read_csv(file_path, delim_whitespace=True, header=None)

x_values = data.iloc[:, 0]
y_values = data.iloc[:, 1:]-fermi_eV
# Convert x and y values to numpy arrays
npx = np.array(x_values)
npy = np.array(y_values)

###########################################################
####################load vasp data################
###########################################################
from jarvis.io.vasp.outputs import Vasprun
vrun = Vasprun('vasprun.xml')
spin = 0  # Assuming spin is predefined
ef=vrun.efermi
kpoints = vrun.kpoints.kpts  # Extract k-points for clarity
eigs = np.array(vrun.eigenvalues[spin][0, :, 0])  # First k-point's eigenvalues

# Iterate through k-points and eigenvalues, starting from the second k-point
for kp, j in enumerate(vrun.eigenvalues[spin][1:], start=1):
    eigs = np.column_stack((eigs, j[:, 0]))  # Stack eigenvalues for each k-point

eigs_ef=eigs-ef


info = vrun.get_bandstructure(kpoints_file_path='KPOINTS',plot=True)
kpts=info['kp_labels_points']
label=info['kp_labels']

plt.clf()
the_grid=GridSpec(1,2)
plt.rcParams.update({'font.size':14})
plt.figure(figsize=(10,4))

plt.subplot(the_grid[0,0])
plt.title('(a)')                
plt.ylabel('Energy(eV)')
plot=True
min_arr = []
dd={}
energy_tol=4
kpoints=eigs_ef.shape[1]
erange = [-energy_tol, energy_tol]
for k in range((kpoints)-1):
            for n in eigs_ef.T[k]:
                diff_arr = []
                if n > erange[0] and n < erange[1]:
                    for v in npy[k]:
                        diff = abs(n - v)
                        diff_arr.append(diff)
                if diff_arr != []:
                    tmp = np.min(diff_arr)
                    dd.setdefault(n,tmp)
                    min_arr.append(tmp)
maxdiff = "na"
if min_arr != []:
            # print ('min_arr',min_arr)
            print("MAX diff", max(min_arr))
            maxdiff = max(min_arr)
print("maxdiff", maxdiff)
if plot:
            for i, ii in enumerate(npy.T):
                plt.plot(ii, color="b")
            for i, ii in enumerate(eigs_ef):
                plt.plot(ii, color="r")
            plt.ylim([-energy_tol, energy_tol])
            #if kp_labels_points != [] and kp_labels != []:
            #    plt.xticks(kp_labels_points, kp_labels)
#plt.show()
plt.xlim(kpts[0],kpts[-1])
plt.xticks(kpts, label)
plt.subplot(the_grid[0,1])
plt.title('(b)')
plt.plot(list(dd.values()),list(dd.keys()),'.')
plt.ylabel('Energy(eV)')
plt.xlabel('DFTB-DFT ($\delta$) (eV)')
plt.savefig('compare_dftbp_dist.png',bbox_inches='tight')
#plt.show()

/var/folders/4h/x0ppqvf13bl263984m6vck7h002fvn/T/ipykernel_123/71542901.py:29: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(file_path, delim_whitespace=True, header=None)


gap= 0.6555999999999997
MAX diff 0.9725775299999997
maxdiff 0.9725775299999997


In [32]:
y_values

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,-13.1282,-1.1372,-1.1372,-1.1372,1.5988,1.5988,1.5988,2.4998,7.7528,7.7528,21.6428,21.6428,21.6428,73.8528,73.8528,107.8928,107.8928,107.8928
1,-13.1232,-1.2192,-1.1722,-1.1722,1.5838,1.6458,1.6458,2.6028,7.6998,7.7618,21.6348,21.6348,21.7478,73.2368,73.6468,107.3548,107.7508,107.7508
2,-13.1082,-1.4412,-1.2692,-1.2692,1.5428,1.7828,1.7828,2.8858,7.5508,7.7878,21.6138,21.6138,22.0578,71.4458,73.0128,105.8808,107.3268,107.3278
3,-13.0822,-1.7552,-1.4132,-1.4132,1.4758,1.9928,1.9928,3.2928,7.3308,7.8308,21.5758,21.5758,22.5578,68.6408,71.9188,103.8228,106.6268,106.6268
4,-13.0452,-2.1202,-1.5872,-1.5872,1.3858,2.2588,2.2588,3.7598,7.0818,7.8928,21.5208,21.5208,23.2258,65.0538,70.3468,101.5938,105.6598,105.6598
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,-9.3362,-9.1202,-4.2312,-3.7882,0.3008,1.0378,10.1088,10.7878,11.1898,11.6548,16.3118,17.5628,28.7008,30.1238,54.3078,57.1078,77.3998,83.5218
160,-9.2942,-9.1742,-4.0902,-3.8202,0.2588,0.6918,10.7508,11.2478,11.6038,11.9258,15.6068,16.4668,28.5428,29.3468,55.9798,57.6328,79.2728,82.8538
161,-9.2662,-9.2132,-3.9712,-3.8432,0.2288,0.4288,11.4048,11.6888,11.9278,12.1298,14.9928,15.4898,28.4008,28.7608,57.2558,58.0178,80.7408,82.3788
162,-9.2492,-9.2362,-3.8902,-3.8562,0.2108,0.2618,12.0028,12.0528,12.1358,12.2578,14.5258,14.6978,28.3038,28.3948,58.0568,58.2518,81.6778,82.0938


In [2]:
fermi_eV

-3.1238

In [33]:
import pandas as pd
import numpy as np

# Load the band structure data
band_file_path = "band_tot.dat"
band_data = pd.read_csv(band_file_path, delim_whitespace=True, header=None)

# Extract energy values (excluding the k-point column)
energies = np.array(band_data.iloc[:, 1:]-fermi_eV+0.3)

# Compute the minimum and maximum of all bands
cbm = np.min(energies[energies > 0])  # Conduction Band Minimum
vbm = np.max(energies[energies < 0])  # Valence Band Maximum

# Band gap
band_gap = cbm - vbm if cbm > vbm else 0.0

cbm, vbm, band_gap

/var/folders/4h/x0ppqvf13bl263984m6vck7h002fvn/T/ipykernel_123/3780077814.py:6: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  band_data = pd.read_csv(band_file_path, delim_whitespace=True, header=None)


(0.2998, -0.8371999999999999, 1.137)

In [31]:
energies

array([[-13.1282,  -1.1372,  -1.1372, ..., 107.8928, 107.8928, 107.8928],
       [-13.1232,  -1.2192,  -1.1722, ..., 107.3548, 107.7508, 107.7508],
       [-13.1082,  -1.4412,  -1.2692, ..., 105.8808, 107.3268, 107.3278],
       ...,
       [ -9.2662,  -9.2132,  -3.9712, ...,  58.0178,  80.7408,  82.3788],
       [ -9.2492,  -9.2362,  -3.8902, ...,  58.2518,  81.6778,  82.0938],
       [ -9.2442,  -9.2442,  -3.8612, ...,  58.3308,  81.9998,  81.9998]])